In [1]:
import pandas as pd
import numpy as np

# Load the training and testing datasets
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display the first few rows of the training and testing datasets
print("Training Data:")
print(train_df.head())
print("\nTesting Data:")
print(test_df.head())

# Display the summary statistics of the training and testing datasets
print("\nTraining Data Summary:")
print(train_df.describe())
print("\nTesting Data Summary:")
print(test_df.describe())

# Distinguish column types
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns

print("\nNumeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Correlation matrix for numeric columns
correlation_matrix = train_df[numeric_cols].corr()
print("\nCorrelation Matrix:")
print(correlation_matrix)

# Visualize the correlation matrix
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Features')
plt.show()

# Visualize the distribution of categorical columns
for col in categorical_cols:
    plt.figure(figsize=(8, 6))
    sns.countplot(data=train_df, x=col)
    plt.title(f'Distribution of {col}')
    plt.show()


Training Data:
    id  bone_length  rotting_flesh  hair_length  color    type
0  472     0.681615       0.529227     0.625242  white   Ghoul
1  170     0.480836       0.407930     0.539005  clear  Goblin
2  189     0.375197       0.742953     0.320764   blue   Ghost
3  861     0.626017       0.172182     0.408422   blue   Ghoul
4   30     0.250770       0.246258     0.554654  black   Ghost

Testing Data:
    id  bone_length  rotting_flesh  hair_length  color    type
0  779     0.516004       0.527508     0.354857  white   Ghoul
1   72     0.523729       0.318483     0.330146  green  Goblin
2   29     0.500197       0.438418     0.532530  clear   Ghoul
3  745     0.417300       0.377595     0.541834  clear  Goblin
4  119     0.515275       0.582627     0.568721  clear  Goblin

Training Data Summary:
               id  bone_length  rotting_flesh  hair_length
count  296.000000   296.000000     296.000000   296.000000
mean   453.574324     0.433147       0.513055     0.525371
std    260.51

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-09-15 07:51:49.671 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['color', 'type'], 'Numeric': ['id', 'bone_length', 'rotting_flesh', 'hair_length'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, MinMaxScale

# Handle missing values
fill_missing = FillMissingValue(features=['bone_length', 'rotting_flesh', 'hair_length'], strategy='mean')
train_df = fill_missing.fit_transform(train_df.copy())
test_df = fill_missing.transform(test_df.copy())

# Encode categorical variables
label_encode = LabelEncode(features=['color', 'type'])
train_df = label_encode.fit_transform(train_df.copy())
test_df = label_encode.transform(test_df.copy())

# Normalize numerical features
min_max_scale = MinMaxScale(features=['bone_length', 'rotting_flesh', 'hair_length'])
train_df = min_max_scale.fit_transform(train_df.copy())
test_df = min_max_scale.transform(test_df.copy())

# Display the preprocessed data
print("Preprocessed Training Data:")
print(train_df.head())
print("\nPreprocessed Testing Data:")
print(test_df.head())


Preprocessed Training Data:
    id  bone_length  rotting_flesh  hair_length  color  type
0  472     0.820910       0.469621     0.566954      6     1
1  170     0.555319       0.310079     0.467303      3     2
2  189     0.415579       0.750734     0.215119      2     0
3  861     0.747365       0.000000     0.316410      2     1
4   30     0.250987       0.097431     0.485387      0     0

Preprocessed Testing Data:
    id  bone_length  rotting_flesh  hair_length  color  type
0  779     0.601839       0.467359     0.254515      6     1
1   72     0.612057       0.192429     0.225960      4     2
2   29     0.580929       0.350180     0.459822      3     1
3  745     0.471273       0.270179     0.470573      3     2
4  119     0.600875       0.539857     0.501642      3     2


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'bone_length', 'rotting_flesh', 'hair_length', 'color', 'type'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from metagpt.tools.libs.data_preprocess import OneHotEncode

# Assuming train_df and test_df are already preprocessed from previous steps

# Separate features and target variable
X_train = train_df.drop(columns=['id', 'type'])
y_train = train_df['type']

X_test = test_df.drop(columns=['id', 'type'])
y_test = test_df['type']

# One-hot encode categorical features
one_hot_encode = OneHotEncode(features=['color'])
X_train = one_hot_encode.fit_transform(X_train.copy())
X_test = one_hot_encode.transform(X_test.copy())

# Initialize and train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.2f}")

# Save predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_df['id'], 'predicted_type': y_pred})
predictions_df.to_csv('monster_type_predictions.csv', index=False)


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Model Accuracy on Test Set: 0.59


In [6]:
# Evaluate the trained model on the preprocessed testing dataset and report the accuracy

# Ensure the testing data is preprocessed in the same way as the training data
X_test = test_df.drop(columns=['id', 'type'])
y_test = test_df['type']

# Transform the testing data using the same preprocessing steps
X_test = one_hot_encode.transform(X_test.copy())

# Use the trained model to make predictions on the preprocessed testing data
y_pred = model.predict(X_test)

# Calculate the accuracy of the model on the test set
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.2f}")

# Save the predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_df['id'], 'predicted_type': y_pred})
predictions_df.to_csv('monster_type_predictions.csv', index=False)


Model Accuracy on Test Set: 0.59


In [7]:
# Load the pre-trained model and the test data
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from metagpt.tools.libs.data_preprocess import OneHotEncode

# Assuming the model is already trained and saved
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Load the test data
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/test.csv'
test_df = pd.read_csv(test_path)

# Preprocess the test data
X_test = test_df.drop(columns=['id', 'type'])
y_test = test_df['type']

# Apply the same preprocessing steps as the training data
X_test = one_hot_encode.transform(X_test.copy())

# Make predictions
y_pred = model.predict(X_test)

# Save the predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_df['id'], 'predicted_type': y_pred})
predictions_df.to_csv('monster_type_predictions.csv', index=False)

# Calculate and print the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.2f}")


Model Accuracy on Test Set: 0.00
